# 00 Setup Data

This notebook prepares data/config/meta artifacts for HA2 CLIP zero-shot experiments.
It is designed to run with minimal differences on local machine and Colab.


## Cell 1 - Runtime profile

Select profile (`local_4060` or `colab_l4`) and initialize resolved paths.


In [ ]:
from pathlib import Path
import json
import os
import platform

RUNTIME_PROFILE = os.environ.get("HA2_RUNTIME", "local_4060")  # local_4060 | colab_l4
DATASET_NAME = "Caltech101"
SEED_MAIN = 314

project_root_override = os.environ.get("HA2_PROJECT_ROOT")
if project_root_override:
    default_project_root = Path(project_root_override)
else:
    cwd = Path.cwd().resolve()
    if (cwd / "ha2.md").exists():
        default_project_root = cwd
    elif cwd.name == "notebooks" and (cwd.parent / "ha2.md").exists():
        default_project_root = cwd.parent
    else:
        default_project_root = cwd

if RUNTIME_PROFILE == "colab_l4":
    PROJECT_ROOT = Path("/content/HA2")
    DATA_ROOT = PROJECT_ROOT / "data"
    ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts"
    FIGURES_ROOT = ARTIFACTS_ROOT / "figures"
    BATCH_HINT = {"ViT-B/32": 256, "ViT-B/16": 128}
    NUM_WORKERS = 2
else:
    PROJECT_ROOT = default_project_root
    DATA_ROOT = PROJECT_ROOT / "data"
    ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts"
    FIGURES_ROOT = ARTIFACTS_ROOT / "figures"
    BATCH_HINT = {"ViT-B/32": 128, "ViT-B/16": 64}
    NUM_WORKERS = 2

cfg = {
    "runtime_profile": RUNTIME_PROFILE,
    "dataset_name": DATASET_NAME,
    "seed_main": SEED_MAIN,
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "artifacts_root": str(ARTIFACTS_ROOT),
    "figures_root": str(FIGURES_ROOT),
    "batch_hint": BATCH_HINT,
    "num_workers": NUM_WORKERS,
    "models": ["ViT-B/32", "ViT-B/16"],
    "simple_template": "a photo of a {CLASS}.",
}

print(json.dumps(cfg, indent=2))
print("Platform:", platform.platform())


## Cell 2 - Imports and environment checks


In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import Caltech101

ROOT = Path(cfg["project_root"])
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from ha2_common import (
    PROMPTS_RAW_URL,
    build_caltech101_mapping,
    download_file,
    ensure_dirs,
    extract_tar,
    extract_zip,
    fetch_text,
    is_valid_zip,
    parse_prompts_dataset,
    set_seed,
    write_json,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__)
print("Torchvision:", __import__("torchvision").__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("Device for later notebooks:", device)

sns.set_theme(style="whitegrid")


## Cell 3 - Prepare directories and save resolved config


In [ ]:
project_root = Path(cfg["project_root"])
data_root = Path(cfg["data_root"])
artifacts_root = Path(cfg["artifacts_root"])
figures_root = Path(cfg["figures_root"])
meta_root = artifacts_root / "meta"
tables_root = artifacts_root / "tables"

ensure_dirs([data_root, artifacts_root, figures_root, meta_root, tables_root])

write_json(artifacts_root / "config_resolved.json", cfg)
print("Saved:", artifacts_root / "config_resolved.json")


## Cell 4 - Download/Load Caltech101 dataset (with fallback)


In [ ]:
set_seed(cfg["seed_main"])

dataset = None
try:
    dataset = Caltech101(root=str(data_root), download=True)
    print("Loaded Caltech101 via torchvision native download.")
except Exception as e:
    print("Torchvision download failed, using fallback zip source.")
    print("Reason:", repr(e))
    fallback_url = "https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip?download=1"
    zip_path = data_root / "caltech-101.zip"
    extract_root = data_root / "caltech_fallback"

    if (not zip_path.exists()) or (not is_valid_zip(zip_path)):
        if zip_path.exists():
            print("Found invalid/incomplete zip, re-downloading:", zip_path)
            zip_path.unlink()
        print("Downloading fallback zip...")
        download_file(fallback_url, zip_path)
    else:
        print("Using existing fallback zip:", zip_path)

    marker_dir = extract_root / "101_ObjectCategories"
    if not marker_dir.exists():
        print("Extracting fallback zip...")
        extract_zip(zip_path, extract_root)

        tar_img = extract_root / "caltech-101" / "101_ObjectCategories.tar.gz"
        tar_ann = extract_root / "caltech-101" / "Annotations.tar"
        if tar_img.exists():
            print("Extracting image tar...")
            extract_tar(tar_img, extract_root)
        if tar_ann.exists():
            print("Extracting annotation tar...")
            extract_tar(tar_ann, extract_root)
    else:
        print("Fallback extraction already present.")

    tv_root = extract_root / "caltech101"
    tv_root.mkdir(parents=True, exist_ok=True)
    import shutil

    src_img = extract_root / "101_ObjectCategories"
    src_ann = extract_root / "Annotations"
    dst_img = tv_root / "101_ObjectCategories"
    dst_ann = tv_root / "Annotations"
    if src_img.exists() and not dst_img.exists():
        shutil.move(str(src_img), str(dst_img))
    if src_ann.exists() and not dst_ann.exists():
        shutil.move(str(src_ann), str(dst_ann))

    dataset = Caltech101(root=str(extract_root), download=False)
    print("Loaded Caltech101 from fallback path:", extract_root)

categories = list(dataset.categories)

print("Dataset length:", len(dataset))
print("Number of categories:", len(categories))
print("First 8 categories:", categories[:8])


## Cell 5 - Pull CLIP prompt classes/templates and build mapping


In [ ]:
prompts_md = fetch_text(PROMPTS_RAW_URL)
prompt_classes, templates_full = parse_prompts_dataset(prompts_md, cfg["dataset_name"])

mapping_ds_to_prompt = build_caltech101_mapping(categories, prompt_classes)
class_names = [mapping_ds_to_prompt[c] for c in categories]
templates_simple = [cfg["simple_template"]]

print("Prompt classes from CLIP prompts:", len(prompt_classes))
print("Templates (ensemble):", len(templates_full))
print("Templates (simple):", templates_simple)
print("Sample mapping:")
for k in categories[:10]:
    print(f"  {k} -> {mapping_ds_to_prompt[k]}")


## Cell 6 - Build label table and class distribution


In [ ]:
y = np.array(dataset.y, dtype=int)
label_df = pd.DataFrame(
    {
        "index": np.arange(len(dataset), dtype=int),
        "label_id": y,
        "dataset_category": [categories[i] for i in y],
        "prompt_class": [class_names[i] for i in y],
    }
)

class_count_df = (
    label_df.groupby(["label_id", "dataset_category", "prompt_class"], as_index=False)
    .size()
    .rename(columns={"size": "n_samples"})
    .sort_values("label_id")
)

print("Class count summary:")
print(class_count_df["n_samples"].describe())
class_count_df.head()


## Cell 7 - Plot class distribution


In [ ]:
plt.figure(figsize=(18, 4))
plt.bar(class_count_df["label_id"], class_count_df["n_samples"], color="#35608d")
plt.title("Caltech101 Samples per Class")
plt.xlabel("Class ID")
plt.ylabel("#Samples")
plt.tight_layout()
dist_fig_path = figures_root / "fig_dataset_class_hist.png"
plt.savefig(dist_fig_path, dpi=160)
plt.show()
print("Saved:", dist_fig_path)


## Cell 8 - Create eval/smoke indices


In [ ]:
eval_indices = np.arange(len(dataset), dtype=int)

rng = np.random.default_rng(cfg["seed_main"])
smoke_indices = []
for label_id in range(len(categories)):
    cls_idx = label_df.loc[label_df["label_id"] == label_id, "index"].to_numpy()
    take = min(10, len(cls_idx))
    chosen = rng.choice(cls_idx, size=take, replace=False)
    smoke_indices.extend(chosen.tolist())
smoke_indices = np.array(sorted(smoke_indices), dtype=int)

print("Eval size:", len(eval_indices))
print("Smoke size:", len(smoke_indices))


## Cell 9 - Build loaders and run shape sanity check


In [ ]:
smoke_subset = Subset(dataset, smoke_indices.tolist())
smoke_loader = DataLoader(
    smoke_subset,
    batch_size=32,
    shuffle=False,
    num_workers=cfg["num_workers"],
)

sample_img, sample_label = smoke_subset[0]
print("Sample type:", type(sample_img), "label:", sample_label)
print("Smoke loader batches:", len(smoke_loader))


## Cell 10 - Visual sanity grid


In [ ]:
ncols, nrows = 6, 3
vis_idx = np.random.default_rng(2026).choice(smoke_indices, size=ncols * nrows, replace=False)

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 8))
for ax, idx in zip(axes.ravel(), vis_idx):
    img, label = dataset[int(idx)]
    ax.imshow(img)
    ax.set_title(class_names[int(label)], fontsize=8)
    ax.axis("off")

plt.tight_layout()
grid_fig_path = figures_root / "fig_dataset_samples_grid.png"
plt.savefig(grid_fig_path, dpi=160)
plt.show()
print("Saved:", grid_fig_path)


## Cell 11 - Programmatic sanity checks


In [ ]:
checks = {}
checks["n_categories_101_torchvision"] = len(categories) == 101
checks["n_prompt_templates_34"] = len(templates_full) == 34
checks["all_labels_in_range"] = bool((y.min() >= 0) and (y.max() < len(categories)))
checks["mapping_complete"] = len(mapping_ds_to_prompt) == len(categories)
checks["class_names_len_match"] = len(class_names) == len(categories)
checks["eval_full"] = len(eval_indices) == len(dataset)
checks["smoke_nonempty"] = len(smoke_indices) > 0

check_df = pd.DataFrame(
    {"check": list(checks.keys()), "pass": list(checks.values())}
)
display(check_df)
if not all(checks.values()):
    raise RuntimeError("One or more setup sanity checks failed.")
print("All sanity checks passed.")


## Cell 12 - Persist setup artifacts


In [ ]:
write_json(meta_root / "class_names.json", class_names)
write_json(meta_root / "templates_simple.json", templates_simple)
write_json(meta_root / "templates_full.json", templates_full)
write_json(meta_root / "mapping_ds_to_prompt.json", mapping_ds_to_prompt)

label_df.to_csv(meta_root / "labels_full.csv", index=False)
pd.DataFrame({"index": eval_indices}).to_csv(meta_root / "eval_indices.csv", index=False)
pd.DataFrame({"index": smoke_indices}).to_csv(meta_root / "smoke_indices.csv", index=False)
class_count_df.to_csv(meta_root / "class_count.csv", index=False)

setup_summary = {
    "dataset_len": int(len(dataset)),
    "n_categories": int(len(categories)),
    "n_templates_full": int(len(templates_full)),
    "n_templates_simple": int(len(templates_simple)),
    "eval_size": int(len(eval_indices)),
    "smoke_size": int(len(smoke_indices)),
}
write_json(artifacts_root / "setup_summary.json", setup_summary)

print("Saved setup artifacts to:", artifacts_root)
print(json.dumps(setup_summary, indent=2))
